# 패키지 설치

In [ ]:
import torch
import sys
import subprocess


def run_pip(command):
    # 공백 기준으로 명령어를 분리하여 리스트로 만듭니다.
    cmd = [sys.executable, "-m", "pip", "install"] + command.split()
    print(f"🔄 실행 중: {' '.join(cmd)}")
    subprocess.check_call(cmd)


print(
    f"🔥 현재 환경 감지 중... Python {sys.version.split()[0]} / PyTorch {torch.__version__}"
)

# 1. PyTorch 2.4.x 환경인지 확인
if "2.4" in torch.__version__:
    print("✅ 최신 PyTorch 2.4.0 환경입니다. H200 최적화 설치를 진행합니다.")

    # 2. pip 업그레이드 & 빌드 도구 설치
    run_pip("--upgrade pip")
    run_pip("packaging ninja")

    # 3. Flash Attention 2 '완제품' 강제 설치
    print("⬇️ Flash Attention 2 (Pre-built Wheel) 다운로드 중...")
    run_pip(
        "https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu123torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"
    )

    # 4. Unsloth 설치 (공백 제거!)
    print("⬇️ Unsloth (H200 Optimized) 설치 중...")
    # [수정됨] @ 앞뒤의 공백을 제거하여 하나의 문자열로 인식하게 함
    run_pip(
        "unsloth[cu124-ampere-torch240]@git+https://github.com/unslothai/unsloth.git"
    )

    # 5. 나머지 의존성 설치
    print("⬇️ 기타 라이브러리 설치 중...")
    run_pip(
        "trl peft accelerate bitsandbytes huggingface_hub python-dotenv pandas openai matplotlib seaborn scipy"
    )

    print("\n🎉 설치가 완벽하게 끝났습니다!")
    print("👉 상단 메뉴의 [Kernel] -> [Restart Kernel]을 누르고 프로젝트를 시작하세요.")

else:
    print(
        f"⚠️ 경고: 선택한 이미지가 PyTorch 2.4.0이 아닙니다. (현재: {torch.__version__})"
    )
    print("RunPod 템플릿에서 'PyTorch 2.4.0'을 선택했는지 다시 확인해주세요.")

In [ ]:
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

# 🚀 1. SFT 학습 코드 (train_sft.py)

In [ ]:
import os
import shutil

print("🧹 디스크 공간 확보 및 경로 재설정 중...")

# 1. 캐시 경로를 볼륨 디스크(/workspace)로 강제 변경
# (이 코드가 실행된 이후부터는 모든 모델이 넓은 곳에 저장됩니다)
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["HF_HUB_CACHE"] = "/workspace/hf_cache"
os.environ["TORCH_HOME"] = "/workspace/torch_cache"

print(f"✅ 캐시 경로 변경 완료: {os.environ['HF_HOME']}")

# 2. 기존 시스템 디스크(Container Disk)에 쌓인 캐시 삭제 (공간 확보용)
# 시스템 디스크가 꽉 차서 에러가 났으므로, 이걸 비워줘야 다른 작업이 가능합니다.
default_cache_dir = os.path.expanduser("~/.cache/huggingface")

if os.path.exists(default_cache_dir):
    try:
        print(f"🗑️ 기존 캐시 삭제 중: {default_cache_dir} ...")
        shutil.rmtree(default_cache_dir)
        print("✨ 시스템 디스크 공간 확보 완료!")
    except Exception as e:
        print(f"⚠️ 삭제 중 오류 발생 (무시 가능): {e}")
else:
    print("👍 기존 캐시가 없거나 이미 깨끗합니다.")

# 3. 현재 디스크 용량 확인
!df -h /workspace

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
from huggingface_hub import login
from dotenv import load_dotenv
import os
import torch

load_dotenv()
login(token=os.getenv("HUGGINGFACE_API_KEY_FOR_WRITE_ONLY"))

MODEL_NAME = "Bllossom/llama-3-Korean-Bllossom-70B"
OUTPUT_DIR = "sft_model_a"
HF_REPO_ID = "YOUR_HF_ID/Patient-AI-SFT-70B"  # [수정 필요]

# 1. 모델 로드 (H200 최적화)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    token=os.getenv("HUGGINGFACE_API_KEY_FOR_READ_ONLY"),
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# 2. 데이터셋
dataset = load_dataset("json", data_files="sft_train_data.jsonl", split="train")
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""


def formatting_prompts_func(examples):
    texts = [
        alpaca_prompt.format(i, inp, out) + tokenizer.eos_token
        for i, inp, out in zip(
            examples["instruction"], examples["input"], examples["output"]
        )
    ]
    return {"text": texts}


dataset = dataset.map(formatting_prompts_func, batched=True)

# 3. 학습 (H200 전용 배치 사이즈 16)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=8,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=16,
        gradient_accumulation_steps=1,
        warmup_ratio=0.1,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        output_dir="sft_checkpoints",
    ),
)

print("🔥 SFT 학습 시작...")
trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
try:
    model.push_to_hub(HF_REPO_ID, token=os.getenv("HUGGINGFACE_API_KEY_FOR_WRITE_ONLY"))
    tokenizer.push_to_hub(
        HF_REPO_ID, token=os.getenv("HUGGINGFACE_API_KEY_FOR_WRITE_ONLY")
    )
    print("✅ SFT 완료 및 업로드!")
except:
    pass

In [ ]:
# [메모리 청소용 셀]
# SFT 끝난 후, 혹은 DPO 끝난 후 다음 단계 넘어가기 전에 실행
import torch
import gc

# 모델 변수 삭제 (변수명은 상황에 맞게)
try:
    del model
    del tokenizer
    del trainer
except:
    pass

# 가비지 컬렉터 & CUDA 캐시 비우기
gc.collect()
torch.cuda.empty_cache()

print("🧹 GPU 메모리 청소 완료! 다음 단계로 넘어가세요.")

In [ ]:
import os
from huggingface_hub import HfApi, login
from dotenv import load_dotenv

# 1. 환경 설정
load_dotenv()
WRITE_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_WRITE_ONLY")

# 2. 업로드 설정 (여기를 반드시 확인하게!)
LOCAL_DIR = "sft_model_a"  # 학습 코드에서 설정한 OUTPUT_DIR 이름
REPO_ID = "hyunNus/Woooly-SFT-70B"  # [수정 필수] 본인 HF 아이디로 변경

# 3. 검증 및 로그인
if not os.path.exists(LOCAL_DIR):
    print(
        f"❌ 오류: '{LOCAL_DIR}' 폴더가 없습니다. 학습이 제대로 완료되었는지 확인하세요."
    )
    exit()

if not WRITE_TOKEN:
    print("❌ 오류: .env 파일에 WRITE 토큰이 없습니다.")
    exit()

print(f"🔑 Hugging Face 로그인 시도...")
login(token=WRITE_TOKEN)

# 4. 강제 업로드 (HfApi 사용 - 모델 로드 없이 파일만 전송)
print(f"🚀 '{LOCAL_DIR}' 폴더를 '{REPO_ID}'로 업로드 시작...")

try:
    api = HfApi()

    # 레포지토리가 없으면 생성 (private=True 추천)
    api.create_repo(repo_id=REPO_ID, exist_ok=True, repo_type="model")

    # 폴더 통째로 업로드
    api.upload_folder(
        folder_path=LOCAL_DIR,
        repo_id=REPO_ID,
        repo_type="model",
        commit_message="Upload SFT LoRA Adapter (Manual Upload)",
    )

    print(f"✅ 업로드 성공! 확인 링크: https://huggingface.co/{REPO_ID}")

except Exception as e:
    print(f"🔥 업로드 실패 원인:\n{e}")
    print("\n[점검 포인트]")
    print("1. REPO_ID에 본인 아이디가 맞는지 확인했나?")
    print("2. .env의 토큰이 'Write' 권한이 있는지 확인했나?")

# 🚀 3. DPO 학습 (train_dpo.py)1

In [ ]:
load_dotenv()
login(token=os.getenv("HUGGINGFACE_API_KEY_FOR_WRITE_ONLY"))

# 바로 돌리는 경우

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer, is_bfloat16_supported
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
import os

load_dotenv()
login(token=os.getenv("HUGGINGFACE_API_KEY_FOR_WRITE_ONLY"))

SFT_MODEL_PATH = "sft_model_a"
DPO_DATA_FILE = "dpo_train_data.jsonl"
OUTPUT_DIR = "final_dpo_model_b"
HF_REPO_ID = "hyunNus/Woooly-SFT+DPO-70B"  # [수정 필요]

# 1. SFT 모델 로드
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL_PATH,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# 2. DPO 어댑터 추가
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# 3. 학습
dataset = load_dataset("json", data_files=DPO_DATA_FILE, split="train")
PatchDPOTrainer()

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    beta=0.1,
    train_dataset=dataset,
    args=DPOConfig(
        per_device_train_batch_size=8,  # DPO는 안전하게 8
        gradient_accumulation_steps=2,
        warmup_ratio=0.1,
        num_train_epochs=1,
        learning_rate=5e-6,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        output_dir="dpo_checkpoints",
    ),
)

print("🔥 DPO 학습 시작...")
dpo_trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
try:
    model.push_to_hub(HF_REPO_ID, token=os.getenv("HUGGINGFACE_API_KEY_FOR_WRITE_ONLY"))
    tokenizer.push_to_hub(
        HF_REPO_ID, token=os.getenv("HUGGINGFACE_API_KEY_FOR_WRITE_ONLY")
    )
    print("✅ DPO 완료 및 업로드!")
except:
    pass

# 기숙사 와서 돌림

In [ ]:
import os
import shutil
from unsloth import FastLanguageModel, PatchDPOTrainer, is_bfloat16_supported
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
import torch

# ==========================================
# 0. [필수] 디스크 경로 설정 (다운로드 에러 방지)
# ==========================================
print("🧹 디스크 설정 및 캐시 경로 변경 중...")
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["HF_HUB_CACHE"] = "/workspace/hf_cache"
os.environ["TORCH_HOME"] = "/workspace/torch_cache"

# ==========================================
# 1. 환경 설정 및 로그인
# ==========================================
load_dotenv()
WRITE_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_WRITE_ONLY")
READ_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_READ_ONLY")

# 토큰 검증
if not WRITE_TOKEN or not READ_TOKEN:
    raise ValueError("❌ .env 파일이 없거나 토큰이 로드되지 않았습니다.")

login(token=WRITE_TOKEN)

# ==========================================
# 2. 설정 변수
# ==========================================
# SFT 학습된 모델 (Hugging Face에서 불러옴)
SFT_MODEL_ID = "hyunNus/Woooly-SFT-70B"

# DPO 학습 데이터 (로컬 파일)
DPO_DATA_FILE = "dpo_train_data.jsonl"

# 결과 저장 설정
OUTPUT_DIR = "final_dpo_model_b"
HF_REPO_ID = "hyunNus/Woooly-SFT+DPO-70B"

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# ==========================================
# 3. SFT 모델 로드 (From Hugging Face)
# ==========================================
print(f"🚀 SFT 모델 다운로드 및 로드 중: {SFT_MODEL_ID}")
# H200이라도 70B는 크므로 4bit 로드 유지
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    token=READ_TOKEN,  # 다운로드 권한
)

# ==========================================
# 4. DPO 어댑터 설정
# ==========================================
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# ==========================================
# 5. 학습 실행
# ==========================================
# 데이터 확인
if not os.path.exists(DPO_DATA_FILE):
    raise FileNotFoundError(
        f"❌ '{DPO_DATA_FILE}' 파일이 없습니다. 파일 경로를 확인하세요."
    )

dataset = load_dataset("json", data_files=DPO_DATA_FILE, split="train")
PatchDPOTrainer()  # 메모리 최적화

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Unsloth는 Ref 모델 없이 학습 가능 (메모리 절약)
    tokenizer=tokenizer,
    beta=0.1,
    train_dataset=dataset,
    args=DPOConfig(
        per_device_train_batch_size=8,  # H200 권장
        gradient_accumulation_steps=2,
        warmup_ratio=0.1,
        num_train_epochs=1,
        learning_rate=5e-6,  # DPO는 SFT보다 낮은 LR 사용
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        output_dir="dpo_checkpoints",
    ),
)

print("🔥 DPO 학습 시작...")
dpo_trainer.train()

# ==========================================
# 6. 저장 및 업로드
# ==========================================
print(f"💾 최종 모델 로컬 저장 중: {OUTPUT_DIR}")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

try:
    print(f"☁️ Hugging Face 업로드 중: {HF_REPO_ID}")
    model.push_to_hub(HF_REPO_ID, token=WRITE_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=WRITE_TOKEN)
    print("✅ DPO 완료 및 업로드 성공! 수고하셨습니다!")
except Exception as e:
    print(f"⚠️ 업로드 실패 (로컬 저장은 완료됨): {e}")
    print("👉 터미널에서 'huggingface-cli upload' 명령어로 수동 업로드를 시도하세요.")

# 볼륨 디스크로 이동

In [ ]:
import os
import sys

# ==========================================
# 0. [가장 중요] 라이브러리 임포트 전 경로 강제 설정
# ==========================================
# Python 스크립트가 실행되자마자 환경 변수부터 박아둬야
# 나중에 import 되는 라이브러리들이 이 경로를 인식합니다.
print("🧹 디스크 경로를 /workspace로 강제 설정 중...")

# 캐시 저장소 (모델 다운로드, 데이터셋 캐시 등)
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["HF_HUB_CACHE"] = "/workspace/hf_cache"
os.environ["TORCH_HOME"] = "/workspace/torch_cache"

# ==========================================
# 1. 라이브러리 임포트 (환경변수 설정 후)
# ==========================================
import shutil
from unsloth import FastLanguageModel, PatchDPOTrainer, is_bfloat16_supported
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
import torch

# ==========================================
# 2. 환경 설정 및 로그인
# ==========================================
load_dotenv()
WRITE_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_WRITE_ONLY")
READ_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_READ_ONLY")

if not WRITE_TOKEN or not READ_TOKEN:
    raise ValueError("❌ .env 파일이 없거나 토큰이 로드되지 않았습니다.")

login(token=WRITE_TOKEN)

# ==========================================
# 3. 경로 및 변수 설정 (전부 /workspace 기준)
# ==========================================
# SFT 학습된 모델
SFT_MODEL_ID = "hyunNus/Woooly-SFT-70B"

# 작업의 기준이 되는 루트 디렉토리
WORKSPACE_ROOT = "/workspace"

# DPO 학습 데이터 (파일이 /workspace 안에 있다고 가정)
# 만약 현재 폴더에 있다면 그대로 두셔도 되지만, 명시적인 게 안전합니다.
DPO_DATA_FILE = os.path.join(WORKSPACE_ROOT, "dpo_train_data.jsonl")

# [중요] 체크포인트 저장 경로 (중간 저장)
CHECKPOINT_DIR = os.path.join(WORKSPACE_ROOT, "dpo_checkpoints")

# [중요] 최종 결과 저장 경로
OUTPUT_DIR = os.path.join(WORKSPACE_ROOT, "final_dpo_model_b")

HF_REPO_ID = "hyunNus/Woooly-SFT+DPO-70B"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# ==========================================
# 4. SFT 모델 로드
# ==========================================
print(f"🚀 SFT 모델 다운로드 및 로드 중: {SFT_MODEL_ID}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    token=READ_TOKEN,
)

# ==========================================
# 5. DPO 어댑터 설정
# ==========================================
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# ==========================================
# 6. 학습 실행
# ==========================================
# 데이터 파일 존재 확인 (경로 수정됨)
if not os.path.exists(DPO_DATA_FILE):
    # 혹시 몰라 현재 경로에서도 찾아봄
    if os.path.exists("dpo_train_data.jsonl"):
        DPO_DATA_FILE = "dpo_train_data.jsonl"
        print(
            f"⚠️ /workspace 경로에 파일이 없어 현재 경로의 파일을 사용합니다: {DPO_DATA_FILE}"
        )
    else:
        raise FileNotFoundError(f"❌ '{DPO_DATA_FILE}' 파일이 없습니다.")

dataset = load_dataset("json", data_files=DPO_DATA_FILE, split="train")
PatchDPOTrainer()

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    beta=0.1,
    train_dataset=dataset,
    args=DPOConfig(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        warmup_ratio=0.1,
        num_train_epochs=1,
        learning_rate=5e-6,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        # [수정됨] 절대 경로로 지정하여 /workspace에 저장
        output_dir=CHECKPOINT_DIR,
        # [추가됨] 용량 폭발 방지: 가장 최근 체크포인트 1개만 남기고 삭제
        save_total_limit=1,
        # [추가됨] 저장 전략: epoch마다 저장 (혹은 steps로 변경 가능)
        save_strategy="epoch",
    ),
)

print("🔥 DPO 학습 시작...")
dpo_trainer.train()

# ==========================================
# 7. 저장 및 업로드
# ==========================================
print(f"💾 최종 모델 로컬 저장 중: {OUTPUT_DIR}")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

try:
    print(f"☁️ Hugging Face 업로드 중: {HF_REPO_ID}")
    model.push_to_hub(HF_REPO_ID, token=WRITE_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=WRITE_TOKEN)
    print("✅ DPO 완료 및 업로드 성공! 수고하셨습니다!")
except Exception as e:
    print(f"⚠️ 업로드 실패 (로컬 저장은 {OUTPUT_DIR}에 완료됨): {e}")
    print("👉 터미널에서 'huggingface-cli upload' 명령어로 수동 업로드를 시도하세요.")

In [ ]:
# [메모리 청소용 셀]
# SFT 끝난 후, 혹은 DPO 끝난 후 다음 단계 넘어가기 전에 실행
import torch
import gc

# 모델 변수 삭제 (변수명은 상황에 맞게)
try:
    del model
    del tokenizer
    del trainer
except:
    pass

# 가비지 컬렉터 & CUDA 캐시 비우기
gc.collect()
torch.cuda.empty_cache()

print("🧹 GPU 메모리 청소 완료! 다음 단계로 넘어가세요.")

In [ ]:
# [Cell 5] 메모리 대청소 (Evaluation 전 필수)
import torch
import gc

# 1. 학습 관련 객체 삭제
# (변수명이 다를 수 있으니 try-except로 안전하게 삭제)
objects_to_delete = ["model", "tokenizer", "trainer", "dpo_trainer", "dataset"]

for obj_name in objects_to_delete:
    if obj_name in globals():
        del globals()[obj_name]
        print(f"🗑️ Deleted: {obj_name}")

# 2. 가비지 컬렉션 및 CUDA 캐시 비우기
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# 3. 메모리 상태 확인
print(f"✨ GPU Memory Cleaned!")
print(f"Current Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Current Memory Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# ⚖️ 4. 통합 벤치마크 평가 (evaluate_all.py)
Standard Metrics와 K-PatientBench를 동시에 평가하고 시각화합니다.

In [ ]:
import os, json, torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from tqdm import tqdm
from openai import OpenAI
from unsloth import FastLanguageModel
from dotenv import load_dotenv
from math import pi

load_dotenv()
OPENAI_CLIENT = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
READ_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_READ_ONLY")
MODELS = {"Baseline (SFT)": "sft_model_a", "Ours (DPO)": "final_dpo_model_b"}

# --- 1. 벤치마크 데이터 생성 ---
EVAL_DATA = [
    # Track A: Standard
    {
        "group": "Standard",
        "metric": "Prompt-to-Line",
        "phase": "All",
        "persona": "김철수, 32세, 작가",
        "query": "본인 소개 부탁해요.",
        "criteria": "32세 작가 설정 유지",
    },
    {
        "group": "Standard",
        "metric": "Line-to-Line",
        "phase": "All",
        "persona": "방금 밥 먹었다고 함",
        "query": "식사는 하셨나요?",
        "criteria": "먹었다고 답해야 함",
    },
    # Track B: K-PatientBench
    {
        "group": "K-Patient",
        "metric": "Resistance (S1)",
        "phase": "Early",
        "persona": "상담 불신, 방어적",
        "query": "오시는 길 힘들지 않으셨나요?",
        "criteria": "단답형, 귀찮음, 냉소적 태도",
    },
    {
        "group": "K-Patient",
        "metric": "Resistance (S1)",
        "phase": "Early",
        "persona": "무기력, 외부 탓",
        "query": "요즘 가장 힘든 점은?",
        "criteria": "환경 탓, 남 탓, 회피",
    },
    {
        "group": "K-Patient",
        "metric": "Change_Talk (S5)",
        "phase": "Late",
        "persona": "변화 의지 있음",
        "query": "지난주 산책은 어떠셨나요?",
        "criteria": "시도함(Ability), 의지(Desire) 표현",
    },
    {
        "group": "K-Patient",
        "metric": "Change_Talk (S5)",
        "phase": "Late",
        "persona": "통찰 획득",
        "query": "앞으로의 목표는?",
        "criteria": "구체적이고 긍정적인 계획(Plan)",
    },
]


# --- 2. 추론 및 채점 ---
def run_eval():
    results = []
    alpaca = "Below is an instruction...\n\n### Instruction:\n당신은 내담자입니다. 상황에 맞춰 대답하세요.\n\n### Input:\n[상황: {phase}]\n설정: {persona}\n상담사: {query}\n\n### Response:\n"

    # 모델별 추론
    for name, path in MODELS.items():
        print(f"\n🚀 Testing {name}...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=path, max_seq_length=2048, load_in_4bit=True, token=READ_TOKEN
        )
        FastLanguageModel.for_inference(model)

        for item in tqdm(EVAL_DATA):
            prompt = alpaca.format(
                phase=item["phase"], persona=item["persona"], query=item["query"]
            )
            inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
            outputs = model.generate(**inputs, max_new_tokens=150, use_cache=True)
            res = (
                tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
                .split("### Response:\n")[-1]
                .strip()
            )

            # 채점 (Judge)
            judge_prompt = f"평가 기준: {item['metric']} / 정답 기준: {item['criteria']} / 답변: {res}\n\n1~5점 평가 및 이유를 JSON으로 출력: {{'score': 점수, 'reason': '이유'}}"
            try:
                api_res = OPENAI_CLIENT.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": judge_prompt}],
                    response_format={"type": "json_object"},
                )
                score = json.loads(api_res.choices[0].message.content)
                results.append(
                    {**item, "model": name, "response": res, "score": score["score"]}
                )
            except:
                results.append({**item, "model": name, "response": res, "score": 0})

        del model, tokenizer
        torch.cuda.empty_cache()

    return pd.DataFrame(results)


# --- 3. 시각화 (Radar Chart) ---
def visualize(df):
    df_k = df[df["group"] == "K-Patient"]
    pivot = df_k.groupby(["model", "metric"])["score"].mean().unstack()
    categories = list(pivot.columns)

    angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    plt.xticks(angles[:-1], categories)

    colors = {"Baseline (SFT)": "grey", "Ours (DPO)": "red"}
    for model in pivot.index:
        values = pivot.loc[model].tolist()
        values += values[:1]
        ax.plot(angles, values, linewidth=2, label=model, color=colors[model])
        ax.fill(angles, values, color=colors[model], alpha=0.25)

    plt.legend(loc="upper right", bbox_to_anchor=(0.1, 0.1))
    plt.savefig("final_radar_chart.png", dpi=300)
    print("📊 Radar Chart 저장 완료")


if __name__ == "__main__":
    df = run_eval()
    df.to_csv("final_results.csv", index=False)
    visualize(df)
    print("🎉 전체 프로젝트 완료!")

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from openai import OpenAI
from unsloth import FastLanguageModel
from dotenv import load_dotenv
from math import pi

# 1. 환경 설정
load_dotenv()
OPENAI_CLIENT = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
READ_TOKEN = os.getenv("HUGGINGFACE_API_KEY_FOR_READ_ONLY")

# 🚨 [수정됨] 로컬 경로 대신 Hugging Face ID 사용
MODELS = {
    "Baseline (SFT)": "hyunNus/Woooly-SFT-70B",
    "Ours (DPO)": "hyunNus/Woooly-SFT+DPO-70B",
}

# 2. 평가 데이터 (Standard + K-Patient)
EVAL_DATA = [
    # --- Track A: Standard Metrics (기존 연구) ---
    {
        "group": "Standard",
        "metric": "Prompt-to-Line",
        "phase": "All",
        "persona": "김철수, 32세, 작가",
        "query": "본인 소개 부탁해요.",
        "criteria": "32세 작가 설정 유지",
    },
    {
        "group": "Standard",
        "metric": "Line-to-Line",
        "phase": "All",
        "persona": "방금 밥 먹었다고 함",
        "query": "식사는 하셨나요?",
        "criteria": "먹었다고 답해야 함 (문맥 일치)",
    },
    # --- Track B: K-PatientBench (제안 연구) ---
    {
        "group": "K-Patient",
        "metric": "Resistance",
        "phase": "Early",
        "persona": "우울증 초기, 상담 불신",
        "query": "오시는 길 힘들지 않으셨나요?",
        "criteria": "단답형, 귀찮음, 냉소적 태도",
    },
    {
        "group": "K-Patient",
        "metric": "Resistance",
        "phase": "Early",
        "persona": "무기력, 외부 탓",
        "query": "요즘 가장 힘든 점은?",
        "criteria": "환경 탓, 남 탓, 회피",
    },
    {
        "group": "K-Patient",
        "metric": "Change_Talk",
        "phase": "Late",
        "persona": "변화 의지 있음",
        "query": "지난주 산책은 어떠셨나요?",
        "criteria": "시도함(Ability), 의지(Desire) 표현",
    },
    {
        "group": "K-Patient",
        "metric": "Change_Talk",
        "phase": "Late",
        "persona": "통찰 획득",
        "query": "앞으로의 목표는?",
        "criteria": "구체적이고 긍정적인 계획(Plan)",
    },
]


# 3. 추론 함수
def run_inference(name, path):
    print(f"\n🚀 Testing {name} ({path})...")
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=path, max_seq_length=2048, load_in_4bit=True, token=READ_TOKEN
        )
    except Exception as e:
        print(f"❌ 모델 로드 실패: {path}. 경로를 확인하세요. ({e})")
        return []

    FastLanguageModel.for_inference(model)

    results = []
    alpaca = "Below is an instruction...\n\n### Instruction:\n당신은 내담자입니다. 상황에 맞춰 대답하세요.\n\n### Input:\n[상황: {phase}]\n설정: {persona}\n상담사: {query}\n\n### Response:\n"

    for item in tqdm(EVAL_DATA, desc=f"Generating ({name})"):
        prompt = alpaca.format(
            phase=item["phase"], persona=item["persona"], query=item["query"]
        )
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=150, use_cache=True)
        res = (
            tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
            .split("### Response:\n")[-1]
            .strip()
        )
        results.append({**item, "model": name, "response": res})

    del model, tokenizer
    torch.cuda.empty_cache()
    return results


# 4. 채점 함수 (JSON 파싱 보완)
def run_judge(results):
    print("\n⚖️ Judging...")
    scored_results = []
    for r in tqdm(results):
        prompt = f"평가 기준: {r['metric']} / 정답 기준: {r['criteria']} / 답변: {r['response']}\n\n1~5점 평가 및 이유를 JSON으로 출력: {{'score': 점수, 'reason': '이유'}}"
        try:
            api_res = OPENAI_CLIENT.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
            )
            content = api_res.choices[0].message.content

            # JSON 파싱 강화 (Markdown 제거)
            if content.startswith("```json"):
                content = content.replace("```json", "").replace("```", "")

            score_data = json.loads(content)
            r["score"] = score_data["score"]
            r["reason"] = score_data["reason"]
        except Exception as e:
            print(f"⚠️ 채점 에러: {e}")
            r["score"] = 0
            r["reason"] = "Error"
        scored_results.append(r)
    return pd.DataFrame(scored_results)


# 5. 시각화 (Radar + Bar)
def visualize(df):
    # [1] Radar Chart (K-PatientBench)
    df_k = df[df["group"] == "K-Patient"]
    if not df_k.empty:
        pivot = df_k.groupby(["model", "metric"])["score"].mean().unstack()
        categories = list(pivot.columns)
        angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
        angles += angles[:1]

        fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
        plt.xticks(angles[:-1], categories, size=12)
        colors = {"Baseline (SFT)": "grey", "Ours (DPO)": "red"}

        for model in pivot.index:
            val = pivot.loc[model].tolist()
            val += val[:1]
            ax.plot(
                angles, val, linewidth=2, label=model, color=colors.get(model, "blue")
            )
            ax.fill(angles, val, color=colors.get(model, "blue"), alpha=0.25)

        plt.title("Dynamic Persona Capability (K-PatientBench)", y=1.1, weight="bold")
        plt.legend(loc="upper right", bbox_to_anchor=(0.1, 0.1))
        plt.savefig("final_radar_chart.png", dpi=300)
        print("📊 Radar Chart 저장 완료")

    # [2] Bar Chart (Standard Metrics)
    df_s = df[df["group"] == "Standard"]
    if not df_s.empty:
        plt.figure(figsize=(8, 5))
        sns.barplot(
            data=df_s, x="metric", y="score", hue="model", palette=["grey", "red"]
        )
        plt.title("Standard Consistency Metrics", weight="bold")
        plt.ylim(0, 5.5)
        plt.savefig("final_bar_chart.png", dpi=300)
        print("📊 Bar Chart 저장 완료")


# 6. 메인 실행
if __name__ == "__main__":
    all_res = []
    for name, path in MODELS.items():
        all_res.extend(run_inference(name, path))

    if all_res:
        final_df = run_judge(all_res)
        final_df.to_csv("final_results.csv", index=False, encoding="utf-8-sig")
        visualize(final_df)
        print("\n🎉 전체 평가 완료! CSV와 이미지 파일을 확인하세요.")
        print(final_df.groupby(["model", "group", "metric"])["score"].mean())
    else:
        print("❌ 추론된 결과가 없습니다. 모델 경로를 확인하세요.")